In [1]:
import numpy as np
from jax import numpy as jnp
import jax
import matplotlib.pyplot as plt

from blueprint.qubits import TunableTransmon, AnharmonicOscillator
from blueprint.devices import Device

import os, math

os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.30"  # for 50% of total GPU memory

jax.config.update("jax_default_device", jax.devices("cpu")[0])
# jax.config.update('jax_default_device', jax.devices('cuda')[1])
jax.config.update("jax_enable_x64", True)  # Double
dtype = jnp.complex128
# jax.config.update('jax_enable_x64', False) # Simple
# dtype = jnp.complex64

CUDA backend failed to initialize: Unable to load cuDNN. Is it installed? (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


In [2]:
# units
kHz, MHz, GHz = 1e-6, 1e-3, 1
ns, us, ms, s = 1 * 2 * math.pi, 1e3 * 2 * math.pi, 1e6 * 2 * math.pi, 1e9 * 2 * math.pi

## Qubits

In [3]:
label = "Q1"
charging_energy = 0.24193246329355922 * GHz
josephson_energy = 16.861934403863096 * GHz
charge_cutoff = 100
offset_charge = 0.0
dim = 3

transmon = TunableTransmon(
    label=label,
    charging_energy=charging_energy,
    josephson_energy=josephson_energy,
    offset_charge=offset_charge,
    charge_cutoff=charge_cutoff,
)
transmon.diagonalize(dim)

label = "Q2"
charging_energy = 0.23958546598194805 * GHz
josephson_energy = 18.062900398155875 * GHz
charge_cutoff = 100
offset_charge = 0.0
dim = 3

other_transmon = TunableTransmon(
    label=label,
    charging_energy=charging_energy,
    josephson_energy=josephson_energy,
    offset_charge=offset_charge,
    charge_cutoff=charge_cutoff,
)
other_transmon.diagonalize(dim)

In [4]:
print(transmon.get_hamiltonian())

[[ 0.          0.          0.        ]
 [ 0.          5.45922431  0.        ]
 [ 0.          0.         10.64760171]]


## Device

In [5]:
(
    7 * MHz
    / float(transmon.charge_zpf.real)
    / float(other_transmon.charge_zpf.real)
)

0.004863801980067174

In [6]:
device = Device([transmon, other_transmon])
device.add_capacative_coupling(
    qubits=("Q1", "Q2"),
    label="C_Q1_Q2",
    prefactor=7 * MHz
    / float(transmon.charge_zpf)
    / float(other_transmon.charge_zpf),
)

In [7]:
print(device.get_hamiltonian())

[[ 1.27860142e-33+0.j  7.08050600e-18+0.j -1.41473962e-33+0.j
   1.26406361e-18+0.j  7.00000000e-03+0.j -1.39865390e-18+0.j
  -9.84044247e-34+0.j -5.44933790e-18+0.j  1.08881967e-33+0.j]
 [ 7.08050600e-18+0.j  5.63337804e+00+0.j -9.77062622e-18+0.j
   7.00000000e-03+0.j -1.76050227e-18+0.j -9.65953331e-03+0.j
  -5.44933790e-18+0.j  1.37051025e-33+0.j  7.51972299e-18+0.j]
 [-1.40967283e-33+0.j -9.77062622e-18+0.j  1.09999300e+01+0.j
  -1.39364472e-18+0.j -9.65953331e-03+0.j -8.06370642e-18+0.j
   1.08492014e-33+0.j  7.51972299e-18+0.j  6.27740871e-33+0.j]
 [ 1.26406361e-18+0.j  7.00000000e-03+0.j -1.39865390e-18+0.j
   5.45922431e+00+0.j -9.93435174e-18+0.j  1.98495997e-33+0.j
  -1.74221062e-18+0.j -9.64783280e-03+0.j  1.92771128e-18+0.j]
 [ 7.00000000e-03+0.j -1.76050227e-18+0.j -9.65953331e-03+0.j
  -9.93435174e-18+0.j  1.10926024e+01+0.j  1.37087431e-17+0.j
  -9.64783280e-03+0.j  2.42643308e-18+0.j  1.33133660e-02+0.j]
 [-1.39364472e-18+0.j -9.65953331e-03+0.j -8.06370642e-18+0.j
   

In [8]:
print(device.get_int_hamiltonian()[jnp.abs(device.get_int_hamiltonian())>1e-8].real)

[ 0.007       0.007      -0.00965953 -0.00965953  0.007      -0.00964783
  0.007      -0.00965953 -0.00964783  0.01331337 -0.00965953  0.01331337
 -0.00964783 -0.00964783  0.01331337  0.01331337]


In [9]:
device.is_diagonalized

False

## Drives

### Flux drives

In [10]:
from blueprint.pulses import flux_pulses, coupler_pulses

In [11]:
device["Q1"].add_flux_drive(
    label='flux_Q1_cz_Q1_Q2', flux_pulse=flux_pulses.net_zero_transition_flux_pulse
)
device["Q2"].add_flux_drive(
    label="flux_Q2_cz_Q1_Q2", flux_pulse=flux_pulses.net_zero_transition_flux_pulse
)

In [12]:
device["Q1"].drives["flux_Q1_cz_Q1_Q2"].set_params(
    hold_first_voltage=-0.255,
    transition_voltage=0.0,
    half_hold_time=65.15/2,
    transition_time=2.414131062723352,
    buffer_start=1.9349107142857144e1,
    buffer_end=1.9349107142857144e1,
    flux_per_volt=9.817191999999999,
    gaussian_filter_sigma=0.5,
)

device["Q2"].drives["flux_Q2_cz_Q1_Q2"].set_params(
    hold_first_voltage=0.4163522432365655,
    transition_voltage=0.0,
    half_hold_time=65.15 / 2,
    transition_time=2.414131062723352,
    buffer_start=1.9349107142857144e1,
    buffer_end=1.9349107142857144e1,
    flux_per_volt=9.90753,
    gaussian_filter_sigma=0.5,
)

### Coupler drive

In [13]:
# device._couplings["C_Q1_Q2"].add_drive(
#     label="coupler_cz_Q1_Q2", coupling_pulse=coupler_pulses.capacitive_coupling_pulse
# )

In [14]:
from typing import Callable
from functools import partial
from blueprint.base import QuantumSystem
from blueprint.qubits import TunableTransmon


def build_capacitive_coupling_pulse_from_flux_drives(
    transmon_1: TunableTransmon,
    transmon_2: TunableTransmon,
    flux_drive_label_1: str,
    flux_drive_label_2: str,
    flux_callable_1: Callable = flux_pulses.net_zero_transition_flux_pulse,
    flux_callable_2: Callable = flux_pulses.net_zero_transition_flux_pulse,
):
    # Only use the params that are set (i.e. not None) in the Drive object.
    flux_drive_1 = transmon_1.drives[flux_drive_label_1]
    not_none_params_1 = {
        key: val for key, val in flux_drive_1.param_vals.items() if val is not None
    }
    partial_func_flux_1 = partial(flux_callable_1, **not_none_params_1)
    # Same for transmon 2
    flux_drive_2 = transmon_2.drives[flux_drive_label_2]
    not_none_params_2 = {
        key: val for key, val in flux_drive_2.param_vals.items() if val is not None
    }
    partial_func_flux_2 = partial(flux_callable_2, **not_none_params_2)

    return partial(
        coupler_pulses.capacitive_coupling_pulse,
        flux_drive_transmon_1=partial_func_flux_1,
        flux_drive_transmon_2=partial_func_flux_2,
        EJ_1=transmon_1._ej,
        EJ_2=transmon_2._ej,
        EC_1=transmon_1._ec,
        EC_2=transmon_2._ec,
        asymm_1=transmon_1._asymm,
        asymm_2=transmon_2._asymm,
        static_ext_flux_1=transmon_1._ext_flux,
        static_ext_flux_2=transmon_2._ext_flux,
    )

In [15]:
coupler_pulse = build_capacitive_coupling_pulse_from_flux_drives(
    transmon_1=device["Q1"],
    transmon_2=device["Q2"],
    flux_drive_label_1="flux_Q1_cz_Q1_Q2",
    flux_drive_label_2="flux_Q2_cz_Q1_Q2",
    flux_callable_1=flux_pulses.net_zero_transition_flux_pulse,
    flux_callable_2=flux_pulses.net_zero_transition_flux_pulse,
)

In [16]:
device._couplings["C_Q1_Q2"].add_drive(
    label="coupler_cz_Q1_Q2", coupling_pulse=coupler_pulse
)

In [17]:
device._couplings["C_Q1_Q2"]._drives["coupler_cz_Q1_Q2"].set_params(
    frequency_resonator=10.0 * GHz,
    coupling_res_trans_1=100.0 * MHz,
    coupling_res_trans_2=100.0 * MHz,
)

# TODO: make sure the drives are all good.

## Function to build `Hfull: dq.TimeDependentArray` from a device  

In [18]:
import dynamiqs as dq

/home/gene2902/blueprint-env/lib/python3.11/site-packages/qutip/__init__.py:66: UserWarning: The new version of Cython, (>= 3.0.0) is not supported.
  warnings.warn(


In [19]:
def build_dynamiqs_hamiltonian(device: Device):
    dq_hamiltonian = dq.constant(device.get_hamiltonian())
    for qubit in device.qubits:
        drives_iterator = qubit.get_drive_hamiltonian(decompose=True)
        for prefactor, operator in drives_iterator:
            dq_hamiltonian = dq_hamiltonian + dq.modulated(
                prefactor,
                operator,
                # args=(,),
            )
        # for drive_label, drive in qubit.drives.items():
        #     print(f"Adding drive {drive_label}")
        #     for prefactor, operator in drive.decompose():
        #         print(type(prefactor))
        #         dq_hamiltonian = dq_hamiltonian + dq.modulated(
        #             prefactor, 
        #             operator, 
        #             # args=(,),
        #         )
    for coupling_label, coupling in device._couplings.items():
        for drive_label, drive in coupling._drives.items():
            print(f"Adding coupling drive {drive_label}")
            for prefactor, operator in drive.decompose():
                print(type(prefactor))
                dq_hamiltonian = dq_hamiltonian + dq.modulated(
                    prefactor,
                    operator,  
                    # args=(,),
                )
    return dq_hamiltonian

In [20]:
full_hamiltonian = build_dynamiqs_hamiltonian(device=device)

Adding drive flux_Q1_cz_Q1_Q2
<class 'blueprint.base.terms.TimeDependentTerm'>
<class 'blueprint.base.terms.TimeDependentTerm'>
Adding drive flux_Q2_cz_Q1_Q2
<class 'blueprint.base.terms.TimeDependentTerm'>
<class 'blueprint.base.terms.TimeDependentTerm'>
Adding coupling drive coupler_cz_Q1_Q2
<class 'blueprint.base.terms.TimeDependentTerm'>


In [21]:
full_hamiltonian(t=0.0)

TypeError: TimeDependentTerm.__call__() takes 1 positional argument but 2 were given

In [29]:
device.qubits[0].drives

{'flux_Q1_cz_Q1_Q2': <blueprint.drives.drive.Drive at 0x7f31483c5e50>}

In [32]:
device._couplings["C_Q1_Q2"]._drives

{'coupler_cz_Q1_Q2': <blueprint.drives.drive.Drive at 0x7f314f951310>}

In [45]:
from functools import partial

In [51]:
device["Q1"].drives["flux_Q1_cz_Q1_Q2"]

In [50]:
device["Q1"].drives["flux_Q1_cz_Q1_Q2"].param_vals

{'t': None,
 'hold_first_voltage': -0.255,
 'transition_voltage': 0.0,
 'half_hold_time': 32.575,
 'transition_time': 2.414131062723352,
 'buffer_start': 19.349107142857143,
 'buffer_end': 19.349107142857143,
 'flux_per_volt': 9.817191999999999,
 'gaussian_filter_sigma': 0.5}

In [46]:
partial_func = partial(
    flux_pulses.net_zero_transition_flux_pulse,
    hold_first_voltage=-0.255,
    transition_voltage=0.0,
    half_hold_time=65.15 / 2,
    transition_time=2.414131062723352,
    buffer_start=1.9349107142857144e1,
    buffer_end=1.9349107142857144e1,
    flux_per_volt=9.817191999999999,
    gaussian_filter_sigma=0.5,
)

In [48]:
partial_func(20.0)

-2.2618203645575066

In [39]:
device["Q1"].drives["flux_Q1_cz_Q1_Q2"]._prefactor_terms

In [44]:
print(device["Q1"].drives["flux_Q1_cz_Q1_Q2"]._prefactor_terms[0](t=1.0))

0.0


In [31]:
generator = device["Q1"].drives["flux_Q1_cz_Q1_Q2"].eval_prefactors()
generator

<generator object Drive.eval_prefactors at 0x7f0444530a90>

In [36]:
for thing in generator:
    print(thing(1.0))

In [18]:
device._couplings["C_Q1_Q2"]._drives["coupler_cz_Q1_Q2"].set_params(
    transmon_1=transmon,
    transmon_2=other_transmon,
    applied_flux_1
)

['transmon_2',
 'coupling_res_trans_1',
 'applied_flux_1',
 'transmon_1',
 't',
 'frequency_resonator',
 'coupling_res_trans_2',
 'applied_flux_2']

In [ ]:
# dt_ns = 0.5
# cz_pulse_params = {
#     "amplitude": -0.255,
#     "amplitude2": 0.4163522432365655,
#     "aux_pulses_list": [],
#     "buffer_length_end": 1.9349107142857144e1,
#     "buffer_length_start": 1.9349107142857144e1,
#     "gaussian_filter_sigma": 5e-1,
#     "pulse_length": 6.515000000000002e1,
#     "pulse_type": "NZTransitionControlledPulse",
#     "tqg_freq": 4.663912492557016,
#     "tqg_freq2": 4.670220582552401,
#     "trans_length": 2.414131062723352,
#     "V_per_phi0_1": 9.817191999999999,
#     "V_per_phi0_2": 9.90753,
# }